## **Install the required modules**

In [69]:
!pip install -qU \
accelerate==0.31.0 \
peft==0.11.1 \
bitsandbytes==0.43.1 \
transformers==4.41.2 \
trl==0.9.4 \
sentencepiece==0.2.0 \
triton==3.1.0 \
torchvision


**Restart the session after all the modules are installed.**
Restart the session after all the modules are installed.

##**Import the required Libraries**

In [28]:
# Standard Deep Learning and Hardware Framework
import torch
import transformers
import accelerate
import peft
import trl
import torchvision

# Hugging Face Datasets
from datasets import load_dataset

# Hugging Face Transformers & Pipelines
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, TrainingArguments

# PEFT (Parameter-Efficient Fine-Tuning) / LoRA Tools
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model

# TRL (Transformer Reinforcement Learning) Trainers & Configs
from trl import SFTTrainer, DPOConfig, DPOTrainer

from google.colab import drive
import os

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("torchvision:", torchvision.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.5.1+cu124
transformers: 4.41.2
peft: 0.11.1
trl: 0.9.4
torchvision: 0.20.1+cu124
accelerate: 0.31.0


##**Tiny LLama 1.1 Biliion Chat Model - Version 1.0**

###**Before Fine Tuning**

In [3]:
chat_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
#chat_model_name = "meta-llama/Llama-3.1-8B-Instruct"

chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_name)
chat_model = AutoModelForCausalLM.from_pretrained(chat_model_name)

In [4]:
mcq_generator_chat = pipeline(
"text-generation",
model=chat_model,
tokenizer=chat_tokenizer,
return_full_text=False,
max_new_tokens=150,
do_sample=True,
temperature=0.7
)

###**First Try**

In [5]:
context = """
Photosynthesis is a biological process used by plants, algae, and certain bacteria to convert light energy into chemical energy stored in glucose. It occurs mainly in the chloroplasts of plant cells using chlorophyll pigments.
"""
target_answer = "chloroplasts"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

In [6]:
output = mcq_generator_chat(messages)
output

[{'generated_text': 'Question: What is the primary site of photosynthesis in plant cells, and what role does it play in the process of converting light energy into chemical energy stored in glucose?\n\nAnswer: The primary site of photosynthesis in plant cells is the chloroplasts, where chlorophyll pigments absorb light energy and convert it into chemical energy stored in glucose.'}]

In [8]:
output[0]['generated_text']

'Question: What is the primary site of photosynthesis in plant cells, and what role does it play in the process of converting light energy into chemical energy stored in glucose?\n\nAnswer: The primary site of photosynthesis in plant cells is the chloroplasts, where chlorophyll pigments absorb light energy and convert it into chemical energy stored in glucose.'

###**Second Try**

In [9]:
context = """
In operating systems, a deadlock is a situation where a set of processes are blocked because each process is holding a resource and waiting for another resource held by some other process. The Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks.
"""
target_answer = "Banker's algorithm"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

In [10]:
output = mcq_generator_chat(messages)
output

[{'generated_text': "Question: In what context does Banker's algorithm avoid deadlocks in operating systems?\n\nAnswer: Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks in operating systems."}]

In [11]:
output[0]['generated_text']

"Question: In what context does Banker's algorithm avoid deadlocks in operating systems?\n\nAnswer: Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks in operating systems."

**So as we see the chat model generates the multiple choice question from a context and target answer. But the questions generated do not elicit the answer given properly. So it needs to be trained to make the output optimal.**

##**Checking the Training Data**

###**Squad V2 Dataset**

In [18]:
train_dataset = load_dataset('rajpurkar/squad_v2', split='train') #

df = train_dataset.to_pandas()
df.head()

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,"{'text': ['in the late 1990s'], 'answer_start'..."
1,56be85543aeaaa14008c9065,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,"{'text': ['singing and dancing'], 'answer_star..."
2,56be85543aeaaa14008c9066,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...,"{'text': ['2003'], 'answer_start': [526]}"
3,56bf6b0f3aeaaa14008c9601,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?,"{'text': ['Houston, Texas'], 'answer_start': [..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?,"{'text': ['late 1990s'], 'answer_start': [276]}"


###**Mapping Function**

In [19]:
def format_prompt(example, tokenizer):
    # Extract context and target answer safely (handling SQuAD v2 unanswerable questions)
    context = example["context"]
    answers = example.get("answers", {})

    if answers and len(answers.get("text", [])) > 0:
        target_answer = answers["text"][0]
    else:
        target_answer = "None"

    question = example.get("question", "")

    # Construct the conversational messages including system, user, and assistant turns
    messages = [
        {
            "role": "system",
            "content": "You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer."
        },
        {
            "role": "user",
            "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
        },
        {
            "role": "assistant",
            "content": f"Question: {question}\nAnswer: {target_answer}"
        }
    ]

    # Apply TinyLlama's chat template to format the prompt for training
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

##**Model Quantization**

In [20]:
# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,  # Use 4-bit precision model loading
  bnb_4bit_quant_type="nf4",  # Quantization type
  bnb_4bit_compute_dtype=torch.float16,  # Compute dtype
  bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

In [21]:
# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
  chat_model_name,
  device_map="auto",

  # Leave this out for regular SFT
  quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

In [22]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(chat_model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [23]:
tokenized_train_dataset = train_dataset.map(lambda x: format_prompt(x,tokenizer))

In [24]:
tokenized_train_dataset = tokenized_train_dataset.shuffle(seed=42).select(range(30000))

In [25]:
tokenized_train_dataset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'text'],
    num_rows: 30000
})

In [26]:
print(tokenized_train_dataset["text"][2576])

<|system|>
You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer.</s>
<|user|>
Context: Tristan da Cunha /ˈtrɪstən də ˈkuːnjə/, colloquially Tristan, is both a remote group of volcanic islands in the south Atlantic Ocean and the main island of that group. It is the most remote inhabited archipelago in the world, lying 2,000 kilometres (1,200 mi) from the nearest inhabited land, Saint Helena, 2,400 kilometres (1,500 mi) from the nearest continental land, South Africa, and 3,360 kilometres (2,090 mi) from South America. The territory consists of the main island, also named Tristan da Cunha, which has a north–south length of 11.27 kilometres (7.00 mi) and has an area of 98 square kilometres (38 sq mi), along with the smaller, uninhabited Nightingale Islands and the wildlife reserves of Inaccessible and Gough Islands.
Target Answer: None
Generate a question from the given context where the target answer

###**Setup Google Drive for Saving**

In [31]:
# Paths for intermediate checkpoints and final merged model
checkpoint_drive_path = '/content/drive/MyDrive/Fine_Tuned_Models/tiny_llama_mcq_checkpoints'
target_drive_path = "/content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq"

# Create folders if they do not exist
os.makedirs(checkpoint_drive_path, exist_ok=True)
os.makedirs(target_drive_path, exist_ok=True)

###**LoRA Configuration**

In [32]:
# Prepare LoRA Configuration
peft_config = LoraConfig(
  lora_alpha=32,  # LoRA Scaling
  lora_dropout=0.1,  # Dropout for LoRA Layers
  r=64,  # Rank
  bias="none",
  task_type="CAUSAL_LM",
  target_modules=  # Layers to target
  ["q_proj", "v_proj", "k_proj", "o_proj"]
  )

In [34]:
# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

###**Training Configuration**

In [35]:
# Training arguments
training_arguments = TrainingArguments(
  output_dir=checkpoint_drive_path,
  per_device_train_batch_size=4,
  gradient_accumulation_steps=8, #Effective batch size is now 32 (8x4)
  optim="paged_adamw_32bit",
  learning_rate=2e-4,
  #report_to="none", #Turn off wandb reporting
  lr_scheduler_type="cosine",
  num_train_epochs=1,
  logging_steps=10,
  fp16=True,
  gradient_checkpointing=True,
  # Checkpoint Auto-Saving (Protects against sudden Colab disconnections)
  save_strategy="steps",
  save_steps=200, # Saves progress to Google Drive every 200 steps
  save_total_limit=2, # Keeps the 2 latest saves to avoid filling Drive space

  )

In [36]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
  model=model,
  train_dataset=tokenized_train_dataset,
  dataset_text_field="text",
  tokenizer=tokenizer,
  args=training_arguments,
  max_seq_length=512,
  # Leave this out for regular SFT
  peft_config=peft_config,
  )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1965: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.027500
20,1.459400
30,1.368800
40,1.353900
50,1.341500
60,1.336800
70,1.303600


Step,Training Loss
10,2.027500
20,1.459400
30,1.368800
40,1.353900
50,1.341500
60,1.336800
70,1.303600
80,1.347800
90,1.282700
100,1.314700


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


If training fails resume training from a particular checkpoint

In [ ]:
# 4. RESUME FROM CHECKPOINT ***
# This looks into your drive and loads the exact state from step ***
checkpoint_dir = os.path.join(checkpoint_drive_path, "checkpoint-***")
trainer.train(resume_from_checkpoint=checkpoint_dir)

In [ ]:
# Save the final trained LoRA adapters locally first
trainer.model.save_pretrained("TinyLlama-1.1B-mcq")

###**Merge Adapter**

In [ ]:
# Free up VRAM before loading the merge tool
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-mcq",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

###**Using the Merged Model**

In [ ]:
# Use our predefined prompt template
context = """
In operating systems, a deadlock is a situation where a set of processes are blocked because each process is holding a resource and waiting for another resource held by some other process. The Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks.
"""
target_answer = "Banker's algorithm"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

# Run our instruction-tuned model
pipe = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7
    )
output = pipe(messages)
output

In [ ]:
output[0]['generated_text']

###**Save and Download the Merged Model to your Local Hard drive**

In [ ]:
# Save directly to your Google Drive
merged_model.save_pretrained(target_drive_path)
tokenizer.save_pretrained(target_drive_path)
print(f"Model and tokenizer safely saved to your Google Drive at: {target_drive_path}")